# Customer-Level Master Table

This notebook builds a one-row-per-customer analytical table from the master order table.

We will:
- inspect the master order table columns
- aggregate order-level metrics by `customer_unique_id`
- keep location context for each customer


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

PROJECT_DIR = Path(r"D:/Data_visualization_design/ecommerce-visual-analytics/data-visualization")
MASTER_ORDER_PATH = PROJECT_DIR / "master_order_table.csv"
OUTPUT_PATH = PROJECT_DIR / "customer_level_master_table.csv"

master_order_table = pd.read_csv(
    MASTER_ORDER_PATH,
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "review_creation_date",
        "review_answer_timestamp",
    ],
)

print(f"Loaded master_order_table with {len(master_order_table):,} rows and {len(master_order_table.columns)} columns")
display(master_order_table.head())

Loaded master_order_table with 99,441 rows and 30 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_count,unique_sellers,unique_products,total_freight_value,total_item_value,payment_value_total,payment_installments_total,payment_type_count,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,actual_delivery_days,delivery_delay_days,late_delivery_flag,review_risk_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,8.72,29.99,38.71,3.0,2.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,22.76,118.70,141.46,1.0,1.0,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13.0,-6.0,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,19.22,159.90,179.12,3.0,1.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,9.0,-18.0,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,27.20,45.00,72.20,1.0,1.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.,2017-12-03,2017-12-05 19:21:58,13.0,-13.0,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,8.72,19.90,28.62,1.0,1.0,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17,2018-02-18 13:02:51,2.0,-10.0,False,False


In [2]:
customer_level_master_table = (
    master_order_table
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_orders=("order_id", "nunique"),
        customer_id_count=("customer_id", "nunique"),
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),
        customer_city=("customer_city", lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]),
        customer_state=("customer_state", lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]),
        total_item_value=("total_item_value", "sum"),
        total_freight_value=("total_freight_value", "sum"),
        total_payment_value=("payment_value_total", "sum"),
        avg_payment_value=("payment_value_total", "mean"),
        avg_installments=("payment_installments_total", "mean"),
        avg_review_score=("review_score", "mean"),
        review_count=("review_score", "count"),
        late_delivery_orders=("late_delivery_flag", "sum"),
        avg_delivery_delay_days=("delivery_delay_days", "mean"),
        avg_actual_delivery_days=("actual_delivery_days", "mean"),
        product_diversity=("unique_products", "sum"),
        seller_diversity=("unique_sellers", "sum"),
    )
)

customer_level_master_table["repeat_customer_flag"] = customer_level_master_table["customer_orders"] > 1
customer_level_master_table["late_delivery_rate"] = (
    customer_level_master_table["late_delivery_orders"] / customer_level_master_table["customer_orders"]
)
customer_level_master_table["review_risk_flag"] = customer_level_master_table["avg_review_score"] <= 2
customer_level_master_table["customer_tenure_days"] = (
    customer_level_master_table["last_order_date"] - customer_level_master_table["first_order_date"]
).dt.days

print(f"Customer-Level Master Table rows: {len(customer_level_master_table):,}")
print(f"Customer-Level Master Table columns: {len(customer_level_master_table.columns)}")
display(customer_level_master_table.head())

Customer-Level Master Table rows: 96,096
Customer-Level Master Table columns: 23


,customer_unique_id,customer_orders,customer_id_count,first_order_date,last_order_date,customer_city,customer_state,total_item_value,total_freight_value,total_payment_value,avg_payment_value,avg_installments,avg_review_score,review_count,late_delivery_orders,avg_delivery_delay_days,avg_actual_delivery_days,product_diversity,seller_diversity,repeat_customer_flag,late_delivery_rate,review_risk_flag,customer_tenure_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,2018-05-10 10:56:27,2018-05-10 10:56:27,cajamar,SP,129.90,12.00,141.90,141.90,8.0,5.0,1,0,-5.0,6.0,1.0,1.0,False,0.0,False,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,2018-05-07 11:11:27,2018-05-07 11:11:27,osasco,SP,18.90,8.29,27.19,27.19,1.0,4.0,1,0,-5.0,3.0,1.0,1.0,False,0.0,False,0
2,0000f46a3911fa3c0805444483337064,1,1,2017-03-10 21:05:03,2017-03-10 21:05:03,sao jose,SC,69.00,17.22,86.22,86.22,8.0,3.0,1,0,-2.0,25.0,1.0,1.0,False,0.0,False,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,2017-10-12 20:29:41,2017-10-12 20:29:41,belem,PA,25.99,17.63,43.62,43.62,4.0,4.0,1,0,-12.0,20.0,1.0,1.0,False,0.0,False,0
4,0004aac84e0df4da2b147fca70cf8255,1,1,2017-11-14 19:45:42,2017-11-14 19:45:42,sorocaba,SP,180.00,16.89,196.89,196.89,6.0,5.0,1,0,-8.0,13.0,1.0,1.0,False,0.0,False,0


In [3]:
expected_rows = master_order_table["customer_unique_id"].nunique()
actual_rows = len(customer_level_master_table)

print(f"Expected unique customers: {expected_rows:,}")
print(f"Actual customer-level rows: {actual_rows:,}")
print(f"Duplicate customer_unique_id rows: {customer_level_master_table['customer_unique_id'].duplicated().sum()}")
print("Missing values per column (top 15):")
display(customer_level_master_table.isna().sum().sort_values(ascending=False).head(15))

customer_level_master_table.to_csv(OUTPUT_PATH, index=False)
print(f"Customer-Level Master Table saved to: {OUTPUT_PATH}")
display(customer_level_master_table.head())

Expected unique customers: 96,096
Actual customer-level rows: 96,096
Duplicate customer_unique_id rows: 0
Missing values per column (top 15):


avg_actual_delivery_days    2740
avg_delivery_delay_days     2740
avg_review_score             716
avg_installments               1
avg_payment_value              1
review_risk_flag               0
late_delivery_rate             0
repeat_customer_flag           0
seller_diversity               0
product_diversity              0
late_delivery_orders           0
review_count                   0
customer_unique_id             0
customer_orders                0
total_payment_value            0
dtype: int64

Customer-Level Master Table saved to: D:\Data_visualization_design\ecommerce-visual-analytics\data-visualization\customer_level_master_table.csv


,customer_unique_id,customer_orders,customer_id_count,first_order_date,last_order_date,customer_city,customer_state,total_item_value,total_freight_value,total_payment_value,avg_payment_value,avg_installments,avg_review_score,review_count,late_delivery_orders,avg_delivery_delay_days,avg_actual_delivery_days,product_diversity,seller_diversity,repeat_customer_flag,late_delivery_rate,review_risk_flag,customer_tenure_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,2018-05-10 10:56:27,2018-05-10 10:56:27,cajamar,SP,129.90,12.00,141.90,141.90,8.0,5.0,1,0,-5.0,6.0,1.0,1.0,False,0.0,False,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,2018-05-07 11:11:27,2018-05-07 11:11:27,osasco,SP,18.90,8.29,27.19,27.19,1.0,4.0,1,0,-5.0,3.0,1.0,1.0,False,0.0,False,0
2,0000f46a3911fa3c0805444483337064,1,1,2017-03-10 21:05:03,2017-03-10 21:05:03,sao jose,SC,69.00,17.22,86.22,86.22,8.0,3.0,1,0,-2.0,25.0,1.0,1.0,False,0.0,False,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,2017-10-12 20:29:41,2017-10-12 20:29:41,belem,PA,25.99,17.63,43.62,43.62,4.0,4.0,1,0,-12.0,20.0,1.0,1.0,False,0.0,False,0
4,0004aac84e0df4da2b147fca70cf8255,1,1,2017-11-14 19:45:42,2017-11-14 19:45:42,sorocaba,SP,180.00,16.89,196.89,196.89,6.0,5.0,1,0,-8.0,13.0,1.0,1.0,False,0.0,False,0
